# COMP4328 Assignment 2 — verified code and GPU experiments

Use **Runtime → Change runtime type → GPU**, then run these cells in order. Running the training cells uses your selected Colab runtime and credits. The full predefined suite has eight configurations, ten splits each and six logged methods. Short checks are clearly marked as non-reportable.

This notebook reads inputs and backs up outputs **only in your new `Draft Assignment 2 COMP4328` folder**. Training uses `/content`, with periodic backups to `experiment_results_v2` in the new folder. Run only one notebook against that backup location at a time.

Code includes training-only preprocessing, explicit full-gradient matrix revision, valid probability constraints, equal-budget controls, protected/resumable results, and saved checkpoints. The full final results still need to be generated; local checks do not establish final accuracy.

In [1]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing: Runtime > Change runtime type > GPU")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/Draft Assignment 2 COMP4328")
if not DRIVE_DIR.is_dir():
    raise FileNotFoundError("The new Drive folder is not at its verified My Drive location; check the mounted account")
ROOT = Path("/content/comp4328_a2_v2")
ROOT.mkdir(exist_ok=True)

GPU: NVIDIA L4 | PyTorch: 2.11.0+cu130
Mounted at /content/drive


## 1. Copy and validate inputs

The code ZIP contains an empty `data/` folder for submission. The separate supplied dataset archive is copied into runtime storage and unpacked there. If the archive is not in the new Drive folder, the cell offers a direct runtime upload instead. These inputs are verified by SHA-256 before extraction.

In [2]:
CODE_SHA256 = "60d42f26a3d6d340053e114ec3b4cd26f0962a5b45d88654afd21c96ae305c25"
DATA_SHA256 = "e2ce73884b721359571c1f80f6e2d3510b1d21d57b8a1d6d1286f487657b7aff"
CODE_ARCHIVE = DRIVE_DIR / "assignment2_code_verified_v3.zip"

def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def checked_copy(source, destination, expected):
    if sha256(source) != expected:
        raise ValueError(f"Unexpected file contents: {source.name}")
    shutil.copyfile(source, destination)
    if sha256(destination) != expected:
        raise OSError("Runtime copy checksum mismatch")

local_code = ROOT / "source.zip"
checked_copy(CODE_ARCHIVE, local_code, CODE_SHA256)
with zipfile.ZipFile(local_code) as archive:
    for member in archive.namelist():
        target = (ROOT / member).resolve()
        if target != ROOT and ROOT not in target.parents:
            raise ValueError("Unsafe ZIP path")
    archive.extractall(ROOT)

data_source = DRIVE_DIR / "datasets.zip"
if not data_source.is_file():
    from google.colab import files
    print("Select the supplied datasets.zip for a runtime-local upload")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Select exactly one supplied dataset archive")
    data_source = Path(next(iter(uploaded))).resolve()
local_data = ROOT / "datasets.zip"
checked_copy(data_source, local_data, DATA_SHA256)
expected = {"FashionMNIST0.3.npz", "FashionMNIST0.6.npz", "CIFAR.npz"}
found = set()
with zipfile.ZipFile(local_data) as archive:
    for name in archive.namelist():
        basename = Path(name).name
        if basename in expected:
            if basename in found:
                raise ValueError("Duplicate dataset in archive")
            with archive.open(name) as source, (ROOT / "data" / basename).open("wb") as destination:
                shutil.copyfileobj(source, destination)
            found.add(basename)
if found != expected:
    raise ValueError("Dataset archive is incomplete")
sys.path.insert(0, str(ROOT))
from algorithm.main import choose_device
choose_device("cuda")
print("Validated local source and all three datasets")

Validated local source and all three datasets


## 2. Checks and backup helpers

Completed stages are backed up roughly once a minute and again when a command exits. On a fresh runtime, the backup is restored before resuming. An incomplete training stage restarts from its defined initial checkpoint. Code, configuration, dataset and environment must match for resume; use a new experiment name if any of those change. Interrupting a cell may take a short time while its final backup finishes.

In [3]:
from runtime_backup import sync_tree, run_with_backup

BACKUP = DRIVE_DIR / "experiment_results_v2"
BACKUP.mkdir(exist_ok=True)
CHECKS = ROOT / "checks"
CHECKS.mkdir(exist_ok=True)

def restore_if_needed(local, remote):
    if remote.is_dir() and not local.exists():
        print("Restoring", remote.name)
        sync_tree(remote, local)
    # Only task-owned lock files belonging to dead processes may be cleared.
    if local.exists():
        for lock in local.rglob(".running.lock"):
            pid = int(lock.read_text().strip())
            try:
                os.kill(pid, 0)
            except ProcessLookupError:
                lock.unlink()
            else:
                raise RuntimeError(f"An existing runner may still be active (PID {pid}); do not start a second run")

def launch_suite(local, remote, extra):
    restore_if_needed(local, remote)
    command = [sys.executable, "-u", str(ROOT / "algorithm" / "suite.py"),
               "--data-dir", str(ROOT / "data"), "--output", str(local), "--device", "cuda"] + extra
    if (local / "suite.json").is_file():
        command.append("--resume")
    run_with_backup(command, local, remote, CODE_ARCHIVE, interval=60)

subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(ROOT / "tests"), "-v"], check=True, cwd=ROOT)
print("Regression tests passed")

Regression tests passed


## 3. GPU smoke check

This exercises all eight configurations on two small splits and replays all saved test predictions. It must pass before final training. Its scores are diagnostic and cannot be included as final assignment results.

In [4]:
SMOKE = ROOT / "experiments" / "smoke_v1"
launch_suite(SMOKE, BACKUP / "smoke_v1", ["--smoke", "--runs", "2", "--epochs", "2", "--sample-limit", "600", "--test-limit", "300"])
subprocess.run([sys.executable, str(ROOT / "algorithm" / "verify.py"), "--root", str(SMOKE),
                "--data-dir", str(ROOT / "data"), "--replay", "--device", "cuda",
                "--output", str(CHECKS / "gpu_smoke_audit.json")], check=True)
sync_tree(CHECKS, BACKUP / "checks")
print("GPU smoke checks and checkpoint replay passed")

Backup: 2 updated files
Backup: 322 updated files
cifar_estimated_argmax.log:   completed 2/2: {'ce': 34.99999940395355, 'forward': 45.00000178813934, 'reweight': 38.333335518836975, 'continued': 38.333335518836975, 'continued_full': 57.33333230018616, 'trevision': 57.33333230018616}
GPU smoke checks and checkpoint replay passed


## 4. Full final suite

Running this cell starts the full ten-run experiments. Defaults are fixed in advance: Fashion 20 base + 10 refinement epochs; CIFAR 40 base + 20 refinement epochs; all examples; all eight configurations. Each refinement branch has the same extra epoch budget. No clean-test accuracy is used for model or estimator selection.

Set `RUN_FINAL = False` if you only want the smoke check. Progress and backup messages appear about once a minute. Re-running after an interruption resumes verified completed stages; it does not silently overwrite a different experiment.

In [5]:
RUN_FINAL = True
FINAL = ROOT / "experiments" / "final_v1"
if RUN_FINAL:
    launch_suite(FINAL, BACKUP / "final_v1", [])
    print("All final experiments completed")
else:
    print("Final training skipped")

Backup: 2 updated files
Backup: 25 updated files
fashion0.3_given_trimmed.log:   ce         epoch 06/20: val=0.703026 best=5
Backup: 26 updated files
fashion0.3_given_trimmed.log:   forward    epoch 02/20: val=0.642357 best=2
Backup: 23 updated files
fashion0.3_given_trimmed.log:   forward    epoch 17/20: val=0.641647 best=16
Backup: 26 updated files
fashion0.3_given_trimmed.log:   reweight   epoch 12/20: val=0.642885 best=12
Backup: 26 updated files
fashion0.3_given_trimmed.log:   continued  epoch 07/10: val=0.662726 best=4
Backup: 28 updated files
fashion0.3_given_trimmed.log:   continued_full epoch 10/10: val=0.619026 best=5
Backup: 27 updated files
fashion0.3_given_trimmed.log: fashion0.3 given/trimmed seed=8
Backup: 23 updated files
fashion0.3_given_trimmed.log:   ce         epoch 16/20: val=0.850883 best=15
Backup: 25 updated files
fashion0.6_given_trimmed.log:   ce         epoch 20/20: val=1.159249 best=9
Backup: 25 updated files
fashion0.6_given_trimmed.log:   forward    epoch 

## 5. Verify final scores, export figures and save results

The audit recomputes every saved clean-test accuracy from its selected checkpoint on the GPU. Generated tables contain sample standard deviations across ten splits. The report needs the actual CIFAR transition estimate; these are printed below and saved with per-entry uncertainty. Figures and tables are copied to the new Drive folder.

In [6]:
if RUN_FINAL:
    subprocess.run([sys.executable, str(ROOT / "algorithm" / "verify.py"), "--root", str(FINAL),
                    "--data-dir", str(ROOT / "data"), "--replay", "--device", "cuda",
                    "--output", str(CHECKS / "final_audit.json")], check=True)
    import importlib.util
    if importlib.util.find_spec("matplotlib") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib>=3.8,<4"], check=True)
    from algorithm.analyse import export, experiment_files
    ANALYSIS = ROOT / "analysis" / "final_v1"
    export(FINAL, ANALYSIS, plots=True)
    sync_tree(CHECKS, BACKUP / "checks")
    sync_tree(ANALYSIS, BACKUP / "analysis" / "final_v1")
    from IPython.display import Markdown, display
    display(Markdown((ANALYSIS / "tables.md").read_text()))
    for path in experiment_files(FINAL):
        result = json.loads(path.read_text())
        if result["config"]["dataset"] == "cifar":
            print(path.parent.name)
            for key in ("T_hat", "T_revised"):
                print(key, result["summary"][key])
    print("Finished. Results, checkpoints, audit, figures and tables are backed up under:", BACKUP)

# Experiment results

SD is the sample standard deviation across splits (ddof=1).
Smoke/partial results are verification evidence, not final assignment results.

| Experiment | Method | Runs | Accuracy (%) | Final eligible |
|---|---|---:|---:|---|
| cifar_estimated_argmax | ce | 10 | 59.91 ± 4.77 | True |
| cifar_estimated_argmax | forward | 10 | 63.53 ± 4.90 | True |
| cifar_estimated_argmax | reweight | 10 | 62.31 ± 4.16 | True |
| cifar_estimated_argmax | continued | 10 | 62.88 ± 4.23 | True |
| cifar_estimated_argmax | continued_full | 10 | 62.31 ± 4.16 | True |
| cifar_estimated_argmax | trevision | 10 | 62.31 ± 4.16 | True |
| cifar_estimated_trimmed | ce | 10 | 59.91 ± 4.77 | True |
| cifar_estimated_trimmed | forward | 10 | 69.56 ± 1.98 | True |
| cifar_estimated_trimmed | reweight | 10 | 67.35 ± 4.58 | True |
| cifar_estimated_trimmed | continued | 10 | 67.58 ± 1.79 | True |
| cifar_estimated_trimmed | continued_full | 10 | 67.35 ± 4.58 | True |
| cifar_estimated_trimmed | trevision | 10 | 67.35 ± 4.58 | True |
| fashion0.3_estimated_argmax | ce | 10 | 96.71 ± 1.40 | True |
| fashion0.3_estimated_argmax | forward | 10 | 96.81 ± 1.54 | True |
| fashion0.3_estimated_argmax | reweight | 10 | 96.25 ± 1.65 | True |
| fashion0.3_estimated_argmax | continued | 10 | 96.91 ± 0.74 | True |
| fashion0.3_estimated_argmax | continued_full | 10 | 96.34 ± 1.70 | True |
| fashion0.3_estimated_argmax | trevision | 10 | 96.36 ± 1.72 | True |
| fashion0.3_estimated_trimmed | ce | 10 | 96.71 ± 1.40 | True |
| fashion0.3_estimated_trimmed | forward | 10 | 98.84 ± 0.17 | True |
| fashion0.3_estimated_trimmed | reweight | 10 | 98.24 ± 0.32 | True |
| fashion0.3_estimated_trimmed | continued | 10 | 98.34 ± 0.32 | True |
| fashion0.3_estimated_trimmed | continued_full | 10 | 98.63 ± 0.36 | True |
| fashion0.3_estimated_trimmed | trevision | 10 | 98.69 ± 0.15 | True |
| fashion0.3_given_trimmed | ce | 10 | 96.71 ± 1.40 | True |
| fashion0.3_given_trimmed | forward | 10 | 99.00 ± 0.22 | True |
| fashion0.3_given_trimmed | reweight | 10 | 98.61 ± 0.16 | True |
| fashion0.3_given_trimmed | continued | 10 | 98.80 ± 0.27 | True |
| fashion0.3_given_trimmed | continued_full | 10 | 98.86 ± 0.13 | True |
| fashion0.3_given_trimmed | trevision | 10 | 98.86 ± 0.15 | True |
| fashion0.6_estimated_argmax | ce | 10 | 88.64 ± 2.68 | True |
| fashion0.6_estimated_argmax | forward | 10 | 81.52 ± 6.74 | True |
| fashion0.6_estimated_argmax | reweight | 10 | 82.65 ± 8.70 | True |
| fashion0.6_estimated_argmax | continued | 10 | 82.49 ± 6.38 | True |
| fashion0.6_estimated_argmax | continued_full | 10 | 82.65 ± 8.70 | True |
| fashion0.6_estimated_argmax | trevision | 10 | 82.65 ± 8.70 | True |
| fashion0.6_estimated_trimmed | ce | 10 | 88.64 ± 2.68 | True |
| fashion0.6_estimated_trimmed | forward | 10 | 88.30 ± 8.45 | True |
| fashion0.6_estimated_trimmed | reweight | 10 | 90.33 ± 5.51 | True |
| fashion0.6_estimated_trimmed | continued | 10 | 90.00 ± 5.76 | True |
| fashion0.6_estimated_trimmed | continued_full | 10 | 91.71 ± 6.11 | True |
| fashion0.6_estimated_trimmed | trevision | 10 | 92.23 ± 6.31 | True |
| fashion0.6_given_trimmed | ce | 10 | 88.64 ± 2.68 | True |
| fashion0.6_given_trimmed | forward | 10 | 91.76 ± 3.75 | True |
| fashion0.6_given_trimmed | reweight | 10 | 91.98 ± 3.29 | True |
| fashion0.6_given_trimmed | continued | 10 | 91.78 ± 4.19 | True |
| fashion0.6_given_trimmed | continued_full | 10 | 95.82 ± 1.20 | True |
| fashion0.6_given_trimmed | trevision | 10 | 95.74 ± 0.98 | True |

Paired accuracy differences (percentage points):
- cifar_estimated_argmax / paired_revision_minus_continued_accuracy: mean -0.5733, sample SD 1.410787902382627, n=10.
- cifar_estimated_argmax / paired_revision_minus_full_gradient_fixed_T: mean 0.0000, sample SD 0.0, n=10.
- cifar_estimated_argmax / paired_full_gradient_minus_detached_fixed_T: mean -0.5733, sample SD 1.410787902382627, n=10.
- cifar_estimated_trimmed / paired_revision_minus_continued_accuracy: mean -0.2300, sample SD 4.638684440107523, n=10.
- cifar_estimated_trimmed / paired_revision_minus_full_gradient_fixed_T: mean 0.0000, sample SD 0.0, n=10.
- cifar_estimated_trimmed / paired_full_gradient_minus_detached_fixed_T: mean -0.2300, sample SD 4.638684440107523, n=10.
- fashion0.3_estimated_argmax / paired_revision_minus_continued_accuracy: mean -0.5533, sample SD 1.9359619909515338, n=10.
- fashion0.3_estimated_argmax / paired_revision_minus_full_gradient_fixed_T: mean 0.0200, sample SD 0.0632447389419614, n=10.
- fashion0.3_estimated_argmax / paired_full_gradient_minus_detached_fixed_T: mean -0.5733, sample SD 1.9237178104043997, n=10.
- fashion0.3_estimated_trimmed / paired_revision_minus_continued_accuracy: mean 0.3500, sample SD 0.3263149290483276, n=10.
- fashion0.3_estimated_trimmed / paired_revision_minus_full_gradient_fixed_T: mean 0.0667, sample SD 0.3428282701740565, n=10.
- fashion0.3_estimated_trimmed / paired_full_gradient_minus_detached_fixed_T: mean 0.2833, sample SD 0.3645577091398761, n=10.
- fashion0.3_given_trimmed / paired_revision_minus_continued_accuracy: mean 0.0633, sample SD 0.2991540876463233, n=10.
- fashion0.3_given_trimmed / paired_revision_minus_full_gradient_fixed_T: mean 0.0033, sample SD 0.18152313605126508, n=10.
- fashion0.3_given_trimmed / paired_full_gradient_minus_detached_fixed_T: mean 0.0600, sample SD 0.24884834134190373, n=10.
- fashion0.6_estimated_argmax / paired_revision_minus_continued_accuracy: mean 0.1600, sample SD 6.514273384524164, n=10.
- fashion0.6_estimated_argmax / paired_revision_minus_full_gradient_fixed_T: mean 0.0000, sample SD 0.0, n=10.
- fashion0.6_estimated_argmax / paired_full_gradient_minus_detached_fixed_T: mean 0.1600, sample SD 6.514273384524164, n=10.
- fashion0.6_estimated_trimmed / paired_revision_minus_continued_accuracy: mean 2.2267, sample SD 4.461240613669578, n=10.
- fashion0.6_estimated_trimmed / paired_revision_minus_full_gradient_fixed_T: mean 0.5167, sample SD 1.7436485535152078, n=10.
- fashion0.6_estimated_trimmed / paired_full_gradient_minus_detached_fixed_T: mean 1.7100, sample SD 4.609465119084239, n=10.
- fashion0.6_given_trimmed / paired_revision_minus_continued_accuracy: mean 3.9600, sample SD 4.517095457887615, n=10.
- fashion0.6_given_trimmed / paired_revision_minus_full_gradient_fixed_T: mean -0.0867, sample SD 0.838604935328922, n=10.
- fashion0.6_given_trimmed / paired_full_gradient_minus_detached_fixed_T: mean 4.0467, sample SD 4.733834077021862, n=10.


cifar_estimated_argmax
T_hat {'mean': [[0.5462753812215084, 0.2620783810208594, 0.19164623775763218], [0.19331822248626468, 0.529232465038733, 0.2774493124750023], [0.2685759911300332, 0.2525987035209882, 0.4788253053489786]], 'std': [[0.029986313825091607, 0.028316724449006365, 0.03250574230669382], [0.03740820217925456, 0.05814023069444214, 0.051301245935665374], [0.05247878912861369, 0.04921891422408929, 0.03701407450163361]], 'n': 10}
T_revised {'mean': [[0.5462753683328628, 0.26207837760448455, 0.19164623767137529], [0.19331822097301482, 0.5292324632406235, 0.2774493142962456], [0.26857599020004275, 0.25259869992733003, 0.4788252979516983]], 'std': [[0.02998631282752693, 0.02831672454733466, 0.032505742573252436], [0.037408203325165014, 0.058140227845122475, 0.05130124919032698], [0.0524787901083494, 0.049218913058348, 0.037014075347555284]], 'n': 10}
cifar_estimated_trimmed
T_hat {'mean': [[0.43376889475142466, 0.30125415563567426, 0.264976949612901], [0.26877630279016235, 0.4242

## Next step

The source ZIP is the code submission package; the separate dataset archive is not part of that ZIP. Review the final tables and matrix estimates before writing the report. Keep negative results and explain them. Read `METHODS.md` for notation, the revision variant and comparisons to include, and `README.md` for the submission commands.

The code and local CPU checks were verified before this notebook was prepared. Actual Colab/GPU compatibility and final results are established by the smoke and audit cells above, after you execute them.